# CONCLAVE — Phase 1: Multi-Method Clustering

This notebook runs **Phase 1** of the CONCLAVE pipeline on your own dataset:

`normalize → sample → (optional) reduce dimensions → cluster with multiple methods → export cluster heatmaps for manual annotation`

You edit the settings in each "⚙️ config" cell, then run the notebook top to bottom.

**What comes out the other end:** for each clustering method you choose, a cluster-by-marker
heatmap (`.png`) and a blank `annotation_template_<method>.csv` (one row per cluster) that you
fill in by hand with your cell-type calls. That's the hand-off point to **Phase 2**
(consensus voting + relabeling + flagging), which is a separate notebook.

> **FlowSOM / DepecheR note:** these run via external R scripts, bundled with the package --
> their path is auto-detected in Step 4, no copy-pasting needed. You still need R itself, plus
> the FlowSOM/DepecheR R packages, installed separately -- this notebook only wires up the
> Python↔R plumbing, not the R packages themselves. Both are opt-in (set `USE_FLOWSOM`/
> `USE_DEPECHE = True` in Step 4) so nothing breaks if you don't have R set up.

> **Tip:** for a first run, test on a subsample (see `CSV_NROWS` below) before committing to the
> full dataset — Phase 1 involves a UMAP embedding for sampling, which is the slowest step and
> scales with cell count.

## Step 0 — Setup

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import Image, display

from conclave.phase1 import run_annotation_pipeline_with_resume

pd.set_option("display.max_columns", 50)

## Step 1 — Load your CSV

Set `CSV_PATH` to your file. `CSV_NROWS = None` loads everything; set it to a number
(e.g. `20000`) for a quick test run first.

In [ ]:
# ⚙️ config
CSV_PATH = "Melanoma.csv"
CSV_NROWS = None  # e.g. 20000 for a quick test run; None = full dataset

df = pd.read_csv(CSV_PATH, nrows=CSV_NROWS)
print(f"Loaded {df.shape[0]:,} cells x {df.shape[1]} columns")
df.head()

## Step 2 — Choose markers

`ALL_CANDIDATE_MARKERS` below is every numeric column in your file except the obvious
non-marker columns (spatial coordinates, cell/sample IDs) — check it against your actual
panel and adjust `NON_MARKER_COLS` if your file uses different column names for those.

`MARKERS` is the list CONCLAVE will actually use. It's pre-filled with the 37-marker
lineage panel used in the CONCLAVE manuscript for this dataset (phenotyping/lineage-defining
markers only, excluding QC/state markers like Ki67, PD1, functional/exhaustion markers, etc.)
— edit freely to match what you want clustered on.

In [ ]:
# Columns that are metadata/coordinates, not markers -- adjust to match your file
NON_MARKER_COLS = ["OID", "X", "Y", "ID", "AID"]

ALL_CANDIDATE_MARKERS = [c for c in df.columns if c not in NON_MARKER_COLS]
print(f"{len(ALL_CANDIDATE_MARKERS)} candidate marker columns found:")
print(ALL_CANDIDATE_MARKERS)

In [ ]:
# ⚙️ config -- the markers CONCLAVE will cluster on
MARKERS = [
    'CD34', 'CD31', 'CD141', 'PNAd', 'CD25', 'CD14', 'CD1c', 'CK', 'CD21',
    'FoxP3', 'CD23', 'GRB7', 'CD1A', 'Podoplanin', 'CD138', 'CD248', 'CD64', 'CD163',
    'Pax5', 'IRF8', 'CD20', 'CD8', 'CD303', 'LYZ', 'CD16', 'CD2', 'HLADR', 'IRF4', 'CD5',
    'CD79a', 'CD68', 'CD3', 'CD4', 'CD27', 'PRDM1', 'MELANA', 'S100B',
]

missing = [m for m in MARKERS if m not in df.columns]
if missing:
    raise ValueError(f"These markers aren't in your CSV: {missing}")
print(f"Using {len(MARKERS)} markers for clustering.")

## Step 3 — Sample / batch column

If your dataset spans multiple slides or samples, list the column(s) that identify them here.
Normalization is then done *within* each sample/slide (recommended for batch effects) rather
than pooling everything together. Set `SAMPLE_COLS = None` to normalize across all cells as one
group instead.

In [ ]:
# ⚙️ config
SAMPLE_COLS = ["ID"]  # set to None to normalize across the whole dataset as one group

if SAMPLE_COLS:
    missing = [c for c in SAMPLE_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Sample columns not found in your CSV: {missing}")
    print(f"{df[SAMPLE_COLS[0]].nunique()} unique sample(s) in column '{SAMPLE_COLS[0]}'")

## Step 4 — Pipeline configuration

- **`NORMALIZATION`**: `"z-score"` (recommended default), `"lognorm"`, `"minmax"`, `"iqr-zscore"`,
  `"iqr-minmax"`, or `None`. The two IQR-based options Winsorize each marker to its Tukey fences
  (`[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`) before standardizing/rescaling -- a more robust-to-outliers
  alternative to plain z-score/min-max.
- **`SAMPLING_MODE`** / **`SAMPLE_SIZE`**: Phase 1 clusters on a UMAP-tile-stratified subsample
  for speed and to avoid huge clusters dominating; `"stratified-notproportional"` samples
  roughly equally from each region of expression space. `"random"` and `"none"` (full dataset,
  no subsampling) are also available.
- **`CLUSTER_METHODS`**: any of `phenograph`, `kmeans`, `flowsom` (needs `FLOWSOM_RSCRIPT`),
  `minibatchkmeans`, `birch`, `dbscan`, `agglomerative`, `affinity`, `leiden`, `meanshift`,
  `spectral`, `depeche` (needs an R script, see `depeche_rscript` if you use it).
- **`PHENOGRAPH_K`**: defaults to 25 (the value the CONCLAVE manuscript's sensitivity sweep
  settled on).
- **`DR_METHOD`**: `None` clusters directly on the (normalized) marker space, which is what the
  manuscript uses by default -- `"pca"`, `"umap"`, `"pacmap"`, or `"tsne"` (capped at 3
  dimensions) are also available if you want to reduce dimensionality before clustering.

In [ ]:
# ⚙️ config
OUTDIR = "./phase1_output"

NORMALIZATION = "z-score"
SAMPLING_MODE = "stratified-notproportional"
SAMPLE_SIZE = 50_000          # subsample size Phase 1 clusters on; set >= len(df) for no subsampling
N_TILES_PER_AXIS = 4

DR_METHOD = None              # None, "pca", "umap", "pacmap", or "tsne"
DR_N_COMPONENTS = 15

CLUSTER_METHODS = ("phenograph", "kmeans")   # add "flowsom"/"depeche" below to use them
PHENOGRAPH_K = 25
DERIVE_KMEANS_FROM = "phenograph"            # kmeans uses this method's cluster count

# FlowSOM and DepecheR ship with the package as R scripts -- their path is
# auto-detected below, no copy-pasting needed. You still need R itself, plus
# the FlowSOM/DepecheR R packages, installed separately (this only wires up
# the Python<->R plumbing, not the R packages themselves). Point these at a
# different script instead if you want to use your own.
import conclave.r_scripts as _r_scripts
import pathlib as _pathlib
_r_scripts_dir = _pathlib.Path(_r_scripts.__file__).parent

FLOWSOM_RSCRIPT = str(_r_scripts_dir / "flowsom_clustering.R")  # override with your own path if needed
DEPECHE_RSCRIPT = str(_r_scripts_dir / "depeche_clustering.R")  # override with your own path if needed

USE_FLOWSOM = False  # set True once R + the FlowSOM R package are installed
USE_DEPECHE = False  # set True once R + the DepecheR R package are installed

if USE_FLOWSOM:
    CLUSTER_METHODS = tuple(CLUSTER_METHODS) + ("flowsom",)
if USE_DEPECHE:
    CLUSTER_METHODS = tuple(CLUSTER_METHODS) + ("depeche",)

SEED = 42
TOP_N_MARKERS = 15   # markers shown per cluster in the annotation heatmaps

print("Cluster methods:", CLUSTER_METHODS)

## Step 5 — Run Phase 1

This runs: input validation → sanity checks → normalization → sampling → (optional) DR →
multi-method clustering → per-method heatmaps + annotation templates → saves everything to
`OUTDIR`.

Checkpoints are saved as it goes (`resume=True`), so if this notebook is interrupted, re-running
this cell picks up where it left off instead of starting over. Set `FORCE_RESTART = True` below
to ignore any existing checkpoints and start fresh (e.g. after changing the config above).

In [ ]:
FORCE_RESTART = False  # set True to ignore checkpoints and rerun everything from scratch

df_labeled, meta = run_annotation_pipeline_with_resume(
    df=df,
    markers=MARKERS,
    outdir=OUTDIR,
    sample_cols=SAMPLE_COLS,
    normalization=NORMALIZATION,
    sampling=SAMPLING_MODE,
    sample_size=SAMPLE_SIZE,
    n_tiles_per_axis=N_TILES_PER_AXIS,
    dr_method=DR_METHOD,
    dr_n_components=DR_N_COMPONENTS,
    cluster_methods=CLUSTER_METHODS,
    phenograph_k=PHENOGRAPH_K,
    derive_kmeans_from=DERIVE_KMEANS_FROM,
    flowsom_rscript=FLOWSOM_RSCRIPT,
    depeche_rscript=DEPECHE_RSCRIPT,
    top_n_markers=TOP_N_MARKERS,
    seed=SEED,
    resume=not FORCE_RESTART,
    force_restart=FORCE_RESTART,
)

print()
print("Cells clustered:", df_labeled.shape[0])
print("Cluster counts per method:", meta["results"]["cluster_counts"])

## Step 6 — Review outputs & annotate

For each method in `CLUSTER_METHODS`, two things were saved to
`OUTDIR/04_cluster_heatmaps/`:

- `heatmap_topN_ranked_<method>.png` — mean marker expression per cluster, for eyeballing
  what each cluster likely is
- `annotation_template_<method>.csv` — one row per cluster (`cluster_id`, `n_cells`,
  `annotation`) with a blank `annotation` column

**Next step (manual, outside this notebook):** open each `annotation_template_<method>.csv`,
fill in the `annotation` column with your cell-type call for each cluster using the heatmap
(and your own domain knowledge) as a guide, and save it. Those filled-in files are the input to
the Phase 2 notebook (consensus voting across methods + relabeling + flagging), which isn't
covered here.

In [ ]:
heatmap_dir = Path(OUTDIR) / "04_cluster_heatmaps"

for method in CLUSTER_METHODS:
    png = heatmap_dir / f"heatmap_topN_ranked_{method}.png"
    template = heatmap_dir / f"annotation_template_{method}.csv"
    print(f"--- {method} ---")
    print(f"  heatmap:  {png}")
    print(f"  template: {template}")
    if png.exists():
        display(Image(filename=str(png)))

## Reference: all pipeline outputs

Everything below lives under `OUTDIR`:

- `00_sanitycheck/` — input data quality checks (NaNs, constant markers, etc.)
- `01_normalized_full.csv` — full dataset after normalization
- `02_sampled_full.csv` — the UMAP-tile-stratified subsample used for clustering
- `02_dr/dr_matrix.csv` — dimensionality-reduced matrix (if `DR_METHOD` was set)
- `03_clustering_annotation/clustered_subset_with_labels_on_sampled.csv` — subsample with a
  cluster-label column per method
- `04_cluster_heatmaps/` — heatmaps + annotation templates (see Step 6)
- `pipeline_run_config.json` — full record of every parameter used for this run
- `pipeline_log.txt` — full run log